In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.environ['PYTHONUTF8'] = '1'
import numpy as np, warnings
warnings.filterwarnings('ignore')
DATA = '../data300k'      # the working set
DATA100K = '../data'      # the calibrated baseline the pitch quotes

In [2]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from core.feature_store import FeatureStore
from core.truth import TruthVault
from core.model import Adjudicator

store, vault = FeatureStore.load(DATA), TruthVault(DATA)
ho = store.split('holdout')
y  = vault.labels(store.payment_ids(ho))

models = {}
print(f'{"evidence":<18}{"AUC":>8}{"AP":>8}{"Brier":>9}')
print('-'*45)
for blocks in [('local',), ('network',), ('local','network')]:
    m = Adjudicator(blocks).fit(store, vault)
    models[blocks] = m
    p = m.predict(store, ho)
    print(f'{" + ".join(blocks):<18}{roc_auc_score(y,p):>8.4f}{average_precision_score(y,p):>8.4f}{brier_score_loss(y,p):>9.4f}')

p_loc = models[('local',)].predict(store, ho)
p_net = models[('local','network')].predict(store, ho)
print(f'\nAUC lift from network evidence: +{roc_auc_score(y,p_net)-roc_auc_score(y,p_loc):.4f}')
print(f'AP  lift from network evidence: +{average_precision_score(y,p_net)-average_precision_score(y,p_loc):.4f}')

evidence               AUC      AP    Brier
---------------------------------------------


local               0.8682  0.6189   0.1026


network             0.7768  0.6498   0.0926


local + network     0.9134  0.7633   0.0781

AUC lift from network evidence: +0.0452
AP  lift from network evidence: +0.1444


In [3]:
# the same ablation on the smaller calibrated dataset, for comparison
store100, vault100 = FeatureStore.load(DATA100K), TruthVault(DATA100K)
ho100 = store100.split('holdout'); y100 = vault100.labels(store100.payment_ids(ho100))
row = {}
for blocks in [('local',), ('local','network')]:
    m = Adjudicator(blocks).fit(store100, vault100)
    row[blocks] = roc_auc_score(y100, m.predict(store100, ho100))
print(f'100k  ({len(ho100)} holdout cases): local {row[("local",)]:.4f} -> +net {row[("local","network")]:.4f}  lift +{row[("local","network")]-row[("local",)]:.4f}')
print(f'300k  ({len(ho)} holdout cases): local {roc_auc_score(y,p_loc):.4f} -> +net {roc_auc_score(y,p_net):.4f}  lift +{roc_auc_score(y,p_net)-roc_auc_score(y,p_loc):.4f}')
print('\nMore data narrows the gap. That is the honest read.')

100k  (622 holdout cases): local 0.8295 -> +net 0.9060  lift +0.0765
300k  (1775 holdout cases): local 0.8682 -> +net 0.9134  lift +0.0452

More data narrows the gap. That is the honest read.


In [4]:
amounts = np.array([e.amount_inr for e in ho])
print(f'{"thr":>6}  {"model":<14}{"released":>9}{"prec":>7}{"net Rs":>16}')
print('-'*56)
for thr in (0.05, 0.20):
    nets = {}
    for tag, p in [('local', p_loc), ('local+network', p_net)]:
        rel = p < thr
        good = y[rel] == 0
        net = amounts[rel][good].sum() - amounts[rel][~good].sum()
        nets[tag] = net
        print(f'{thr:>6.2f}  {tag:<14}{rel.sum():>9}{good.mean():>7.3f}{net:>16,.0f}')
    print(f'{"":>6}  {"GAP":<14}{"":>9}{"":>7}{nets["local+network"]-nets["local"]:>+16,.0f}')

   thr  model          released   prec          net Rs
--------------------------------------------------------
  0.05  local               817  0.979      33,158,054
  0.05  local+network       931  0.990      47,252,118
        GAP                                +14,094,064
  0.20  local               998  0.965      48,098,819
  0.20  local+network      1155  0.960      55,058,332
        GAP                                 +6,959,513


In [5]:
from core.policy import PolicyConfig
from core.backtest import run
from core.metrics import grade, StepUpModel, calibration_ledger, CAP_GRID
from core.report import best_cap

base, su = PolicyConfig(), StepUpModel()
print(f'{"evidence":<18}{"cap":>7}{"released":>10}{"prec":>8}{"contribution":>16}')
print('-'*60)
res = {}
for blocks in [('local',), ('local','network')]:
    m = models[blocks]
    cap = best_cap(calibration_ledger(store, m, base), vault, base, su)
    g = grade(run(store, m, base, 'holdout').redecide(base.for_merchant(cap)), vault, su)
    res[blocks] = g
    print(f'{" + ".join(blocks):<18}{cap:>7.3f}{g.n_released:>10}{g.precision:>8.3f}{g.net_contribution_inr:>16,.0f}')

gain = res[('local','network')].net_contribution_inr - res[('local',)].net_contribution_inr
print(f'\nnetwork evidence is worth Rs {gain:,.0f} ({100*gain/res[("local",)].net_contribution_inr:+.1f}%)')
print(f'and it releases {res[("local","network")].n_released - res[("local",)].n_released} more orders at HIGHER precision')
print(f'  {res[("local",)].precision:.3f} -> {res[("local","network")].precision:.3f}')

evidence              cap  released    prec    contribution
------------------------------------------------------------


local               0.005       993   0.983      14,656,021


local + network     0.020      1181   0.986      15,331,742

network evidence is worth Rs 675,721 (+4.6%)
and it releases 188 more orders at HIGHER precision
  0.983 -> 0.986
